<a href="https://colab.research.google.com/github/RenaldyNaufalTA/Komputasi-Numerik/blob/main/SPNL_Metode_Bisection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metode Bisection untuk Persamaan Non-Linear

Notebook ini berisi program sederhana untuk mencari akar persamaan non-linear **f(x) = 0** menggunakan **metode bisection (bagi dua)**. Persamaan, interval, dan toleransi dimasukkan melalui `input()`. Setiap cell kode diikuti oleh cell penjelasan.

**Prinsip kerja:**
1. Tentukan interval [a, b] dengan syarat f(a) · f(b) < 0 (ada perubahan tanda, sehingga akar berada di dalam interval).
2. Hitung titik tengah c = (a + b) / 2.
3. Jika f(a) · f(c) < 0, akar ada di [a, c] sehingga b = c. Jika tidak, akar ada di [c, b] sehingga a = c.
4. Ulangi sampai |f(c)| atau lebar interval lebih kecil dari toleransi.

**Analogi:** program ini bekerja seperti permainan tebak angka. Tebak titik tengah, lihat apakah akar ada di kiri atau kanan, buang separuh yang salah, lalu ulangi sampai tebakan cukup dekat.

**Contoh input:**

| Persamaan f(x) | a | b | Toleransi | Hasil akar (±) |
|---|---|---|---|---|
| `x**3 - x - 2` | `1` | `2` | `1e-6` | 1,52138 |
| `cos(x) - x` | `0` | `1` | `1e-6` | 0,73909 |
| `exp(-x) - x` | `0` | `1` | `1e-6` | 0,56714 |
| `x**2 - 2` | `1` | `2` | `1e-6` | 1,41421 |
| `x**2 - 4*x + 3` | `2` | `4` | `1e-6` | 3 |

## 1. Import Library
### Penjelasan Cell 1

Cell ini memanggil tiga "alat bantu":
- **pandas** untuk membuat tabel iterasi.
- **numpy** untuk fungsi matematika (sin, cos, exp, dll.) dan membuat deretan angka untuk grafik.
- **matplotlib** untuk menggambar grafik.

Singkatan `pd`, `np`, dan `plt` hanya agar penulisannya lebih pendek.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Input Persamaan

Tulis persamaan dalam variabel `x` memakai sintaks Python, misalnya:
- `x**3 - x - 2`
- `cos(x) - x`
- `exp(-x) - x`
- `x**2 - 4*x + 3`

Fungsi yang tersedia: `sin`, `cos`, `tan`, `exp`, `log`, `sqrt`, serta konstanta `pi` dan `e`.

### Penjelasan Cell 2

```python
persamaan = input("Masukkan persamaan f(x) = ")
```
Program menanyakan persamaan Anda. Yang diketik disimpan sebagai **teks** (string) di variabel `persamaan`, misalnya `"x**3 - x - 2"`.

```python
fungsi_aman = {"sin": np.sin, "cos": np.cos, ..., "pi": np.pi, "e": np.e}
```
Ini adalah "kamus" yang mengajarkan program arti kata `sin`, `cos`, `exp`, `pi`, dan seterusnya. Tanpa kamus ini, Python tidak tahu bahwa `cos` yang Anda ketik berarti fungsi cosinus.

```python
def f(x):
    return eval(persamaan, {"__builtins__": {}}, {**fungsi_aman, "x": x})
```
Ini bagian terpenting cell ini. Fungsi `f(x)` mengubah **teks** persamaan menjadi **perhitungan sungguhan**.
- `eval(...)` menjalankan teks sebagai rumus. Contoh: jika `x = 2`, teks `"x**3 - x - 2"` dihitung menjadi 4.
- `{"__builtins__": {}}` memblokir perintah Python berbahaya, sehingga hanya rumus matematika yang bisa dijalankan.
- `{**fungsi_aman, "x": x}` memberi tahu `eval` nilai `x` dan arti `sin`, `cos`, dst.

Fungsi `np.sin` dan kawan-kawan dipilih karena bisa menghitung satu angka maupun banyak angka sekaligus. Ini dibutuhkan saat menggambar grafik.

In [ ]:
persamaan = input("Masukkan persamaan f(x) = ")

# Fungsi matematika yang diperbolehkan dalam persamaan
fungsi_aman = {
    "sin": np.sin, "cos": np.cos, "tan": np.tan,
    "exp": np.exp, "log": np.log, "sqrt": np.sqrt,
    "pi": np.pi, "e": np.e
}

def f(x):
    return eval(persamaan, {"__builtins__": {}}, {**fungsi_aman, "x": x})

print(f"Persamaan yang dipakai: f(x) = {persamaan}")

## 3. Fungsi Metode Bisection
### Penjelasan Cell 3

Ini adalah inti program. Bacalah per bagian.

**a) Cek syarat awal**
```python
if f(a) * f(b) >= 0:
    raise ValueError(...)
```
Metode bisection hanya bekerja jika f(a) dan f(b) **berlawanan tanda** (satu positif, satu negatif). Alasannya, kurva harus menyeberangi sumbu x di antara keduanya, sehingga pasti ada akar di dalamnya. Jika hasil kali keduanya positif atau nol, program berhenti dan memberi pesan error.

**b) Siapkan wadah**
```python
tabel = []
c_lama = None
```
`tabel` menyimpan catatan setiap iterasi. `c_lama` menyimpan tebakan sebelumnya untuk menghitung galat (selisih antartebakan). Pada iterasi pertama belum ada tebakan sebelumnya, jadi nilainya `None`.

**c) Perulangan utama**
```python
for i in range(1, max_iter + 1):
    c = (a + b) / 2
    fc = f(c)
```
Di setiap putaran, program mencari **titik tengah** `c` lalu menghitung nilai fungsi di titik itu. Perulangan dibatasi `max_iter` (100) supaya tidak berjalan selamanya.

**d) Catat ke tabel**
```python
tabel.append({"Iterasi": i, "a": a, "b": b, "c": c, ...})
```
Semua nilai pada iterasi itu disimpan sebagai satu baris tabel.

**e) Kriteria berhenti**
```python
if abs(fc) < tol or (b - a) / 2 < tol:
    return c, tabel
```
Program berhenti dan mengembalikan hasil jika salah satu terpenuhi:
- `abs(fc) < tol`: nilai f(c) sudah sangat dekat dengan 0, artinya c sudah hampir menjadi akar.
- `(b - a) / 2 < tol`: interval sudah sangat sempit, jadi posisi akar sudah cukup pasti.

**f) Perkecil interval**
```python
if f(a) * fc < 0:
    b = c
else:
    a = c
```
Ini inti dari "bagi dua":
- Jika f(a) dan f(c) **berlawanan tanda**, akar ada di sisi kiri, sehingga batas kanan dipindah: `b = c`.
- Jika **tidak**, akar ada di sisi kanan, sehingga batas kiri dipindah: `a = c`.

Setiap putaran membuang separuh interval, sehingga interval cepat menyempit.

**Contoh dengan x³ − x − 2, interval [1, 2]:**

| Iterasi | a | b | c | f(c) | Keputusan |
|---|---|---|---|---|---|
| 1 | 1 | 2 | 1,5 | −0,125 (negatif) | sama tanda dengan f(a) → a = 1,5 |
| 2 | 1,5 | 2 | 1,75 | +1,609 (positif) | beda tanda dengan f(a) → b = 1,75 |
| 3 | 1,5 | 1,75 | 1,625 | ... | dan seterusnya |

In [ ]:
def bisection(f, a, b, tol=1e-6, max_iter=100):
    """
    Mencari akar f(x) = 0 dengan metode bisection.

    Parameter:
        f        : fungsi yang dicari akarnya
        a, b     : batas bawah dan batas atas interval
        tol      : toleransi galat
        max_iter : jumlah iterasi maksimum

    Return:
        akar (float) dan tabel iterasi (list of dict)
    """
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) dan f(b) harus berlawanan tanda (f(a)*f(b) < 0). Ganti interval.")

    tabel = []
    c_lama = None

    for i in range(1, max_iter + 1):
        c = (a + b) / 2
        fc = f(c)
        galat = abs(c - c_lama) if c_lama is not None else None

        tabel.append({
            "Iterasi": i,
            "a": a,
            "b": b,
            "c": c,
            "f(a)": f(a),
            "f(b)": f(b),
            "f(c)": fc,
            "Galat": galat
        })

        # Kriteria berhenti
        if abs(fc) < tol or (b - a) / 2 < tol:
            return c, tabel

        # Tentukan interval baru
        if f(a) * fc < 0:
            b = c
        else:
            a = c

        c_lama = c

    print("Peringatan: iterasi maksimum tercapai.")
    return c, tabel

## 4. Input Interval dan Toleransi
### Penjelasan Cell 4

```python
a = float(input("Masukkan batas bawah (a)  : "))
b = float(input("Masukkan batas atas  (b)  : "))
tol = float(input("Masukkan toleransi (mis. 1e-6): "))
```
Tiga nilai ditanyakan, lalu diubah dari teks menjadi angka desimal dengan `float()`. Yang dimaksud toleransi adalah batas ketelitian, dan makin kecil nilainya makin akurat hasilnya.

```python
print(f"f({a}) = {f(a):.6f}")
print(f"f({b}) = {f(b):.6f}")
```
Cell ini menampilkan nilai f(a) dan f(b) supaya Anda bisa langsung memeriksa apakah tandanya berlawanan sebelum program dijalankan.

In [ ]:
a = float(input("Masukkan batas bawah (a)  : "))
b = float(input("Masukkan batas atas  (b)  : "))
tol = float(input("Masukkan toleransi (mis. 1e-6): "))

print(f"\nf({a}) = {f(a):.6f}")
print(f"f({b}) = {f(b):.6f}")

## 5. Jalankan Program
### Penjelasan Cell 5

```python
akar, tabel = bisection(f, a, b, tol)
```
Fungsi `bisection` dipanggil dengan data dari Anda. Hasilnya ada dua, yaitu nilai akar dan tabel catatan iterasi.

```python
print(f"Akar hampiran  : {akar:.8f}")
print(f"f(akar)        : {f(akar):.2e}")
print(f"Jumlah iterasi : {len(tabel)}")
```
Cell ini menampilkan tiga hal:
- Akar, dibulatkan 8 angka di belakang koma.
- f(akar), yang seharusnya mendekati 0 (`.2e` adalah format ilmiah, misalnya `4.27e-06` berarti 0,00000427).
- Jumlah iterasi, yaitu panjang `tabel`.

In [ ]:
akar, tabel = bisection(f, a, b, tol)

print(f"Akar hampiran  : {akar:.8f}")
print(f"f(akar)        : {f(akar):.2e}")
print(f"Jumlah iterasi : {len(tabel)}")

## 6. Tabel Iterasi
### Penjelasan Cell 6

```python
df = pd.DataFrame(tabel)
pd.options.display.float_format = '{:.6f}'.format
df
```
Daftar catatan iterasi diubah menjadi tabel rapi (`DataFrame`), lalu semua angka ditampilkan dengan 6 angka di belakang koma. Baris terakhir `df` membuat Jupyter menampilkan tabelnya.

Dari tabel ini Anda bisa melihat langkah demi langkah bagaimana a dan b makin mendekat dan f(c) makin mendekati 0.

In [ ]:
df = pd.DataFrame(tabel)
pd.options.display.float_format = '{:.6f}'.format
df

## 7. Visualisasi Fungsi dan Akar
### Penjelasan Cell 7

```python
lebar = (b - a) * 0.25
x = np.linspace(a - lebar, b + lebar, 400)
y = f(x)
```
- `lebar` adalah margin 25% agar grafik sedikit lebih lebar dari interval.
- `np.linspace(...)` membuat 400 titik x yang berjarak sama.
- `f(x)` menghitung nilai y untuk semua titik itu sekaligus. Inilah alasan di Cell 2 dipakai fungsi numpy.

```python
plt.plot(x, y, ...)                       # gambar kurva
plt.axhline(0, ...)                       # garis sumbu x (y = 0)
plt.scatter(akar, f(akar), color="red")   # titik merah di posisi akar
```
Hasilnya berupa kurva fungsi, garis hitam di y = 0, dan titik merah tempat kurva memotong garis itu. Titik merah itulah akarnya. Sisanya (`title`, `xlabel`, `grid`, `legend`, `show`) hanya pelengkap tampilan.

In [ ]:
lebar = (b - a) * 0.25
x = np.linspace(a - lebar, b + lebar, 400)
y = f(x)

plt.figure(figsize=(8, 5))
plt.plot(x, y, label=f"f(x) = {persamaan}")
plt.axhline(0, color="black", linewidth=0.8)
plt.scatter(akar, f(akar), color="red", zorder=5, label=f"Akar ≈ {akar:.6f}")
plt.title("Metode Bisection")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True)
plt.legend()
plt.show()

## 8. Grafik Konvergensi
### Penjelasan Cell 8

```python
plt.plot(df["Iterasi"], df["c"], marker="o")
plt.axhline(akar, color="red", linestyle="--", label="Akar")
```
Grafik ini menampilkan nilai tebakan `c` pada tiap iterasi (garis dengan titik-titik), dibandingkan dengan nilai akar akhir (garis merah putus-putus).

Pada awalnya tebakan melompat-lompat, lalu makin lama makin menempel ke garis merah. Pola inilah yang disebut **konvergen**, yaitu hasil makin mendekati jawaban yang benar.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df["Iterasi"], df["c"], marker="o")
plt.axhline(akar, color="red", linestyle="--", label="Akar")
plt.title("Konvergensi Nilai c terhadap Iterasi")
plt.xlabel("Iterasi")
plt.ylabel("c")
plt.grid(True)
plt.legend()
plt.show()